<a href="https://colab.research.google.com/github/Wbunker/google-ai-tutorials/blob/main/ADK_Learning_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Welcome to Your ADK Adventure - Tools & Memory! 🚀

Welcome, Agent Architect! This notebook is your guide to giving your AI agents two essential superpowers: custom tools and conversational memory.

By the end of this adventure, you will be able to:

- **Build a Foundational Agent**: Create a simple but effective AI agent from scratch using the Google Agent Development Kit (ADK).

- **Grant New Skills with Custom Tools**: Teach an agent to perform new tasks by connecting it to external APIs, like a real-time weather service.

- **Create a Team of Agents**: Assemble a multi-agent system where a primary agent can delegate specialized tasks to other agents.

- **Master Conversational Memory**: Understand the critical role of Sessions in enabling agents to remember previous interactions, handle feedback, and carry on a coherent conversation.


Let's get this adventure started!

## Author

HI, I'm Qingyue (Annie) Wang, a developer advocate and AI engineer at **Google**, passionate about helping developers build with AI and cloud technologies :)


If you have questions with this notebook, contact me on [LinkedIn](https://www.linkedin.com/in/anniewangtech/) , [X](https://twitter.com/anniewangtech) or email anniewangtech0510@Gmail.com


```
  (\__/)
  (•ㅅ•)
  /づ  📚      Enjoy learning AI Agents :)
```


-------------
### 🎁 🛑 Important Prerequisite: Setup Your Environment! 🛑 🎁
-----------------------------------------------------------------------------

👉 **Get Your API Key HERE**: [Google AI Studio](https://aistudio.google.com/app/apikey)

 -----------------------------------------------------------------------------


```
 ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️
   /\_/\     /\_/\     /\_/\      /\_/\       /\_/\
  ( ^_^ )   ( -.- )   ( >_< )   ( =^.^= )    ( o_o )             
```


## Part 0: Setup & Authentication 🔑

First things first, let's get all our tools ready. This step installs the necessary libraries and securely configures your Google API key so your agents can access the power of Gemini.

In [1]:
!pip install google-adk google-generativeai -q

# --- Import all necessary libraries ---
import os
import sys
import json
import asyncio
import random
import string
from uuid import uuid4
from typing import Any, List

import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, Markdown, display

# --- ADK, Agent, and Evaluation Components ---
from google.adk.agents import Agent
from google.adk.events import Event
from google.adk.runners import Runner
import google.adk as adk
from google.adk.tools import google_search
from google.adk.sessions import InMemorySessionService, Session
from google.genai import types
from google.genai.types import Content, Part


print("✅ All libraries are ready to go!")



✅ All libraries are ready to go!


### Configure Your API Key
To use Gemini models, you need an API key from [Google AI Studio](https://aistudio.google.com/app/apikey). This section securely collects your key and configures it for the ADK.


In [2]:
# --- API Key Configuration ---
from google.colab import userdata

# Option 1: Use Colab Secrets (recommended)
# Go to the 🔑 icon in the left sidebar, add a secret named GOOGLE_API_KEY
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    print("✅ API key loaded from Colab Secrets.")
except Exception:
    # Option 2: Paste it directly (less secure but fine for learning)
    import getpass
    GOOGLE_API_KEY = getpass.getpass("🔑 Enter your Google AI Studio API key: ")
    print("✅ API key entered manually.")


✅ API key loaded from Colab Secrets.


In [3]:
# --- Set Environment Variables for ADK ---

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"

print(f"✅ API key configured (starts with '{GOOGLE_API_KEY[:6]}...')")
print("✅ Using Google AI Studio (not Vertex AI).")


✅ API key configured (starts with 'AQ.Ab8...')
✅ Using Google AI Studio (not Vertex AI).


---
## Part 1: Your First Agent - The Day Trip Genie 🧞

Meet your first creation! The `day_trip_agent` is a simple but powerful assistant. We're making it a little smarter by teaching it to understand **budget constraints**.

* **Agent**: The brain of the operation, defined by its instructions, tools, and the AI model it uses.
* **Session**: The conversation history. For this simple agent, it's just a container for a single request-response.
* **Runner**: The engine that connects the `Agent` and the `Session` to process your request and get a response.

```
+--------------------------------------------------+
|         Spontaneous Day Trip Agent 🤖            |
|--------------------------------------------------|
|  Model: gemini-2.5-flash                         |
|  Description:                                    |
|   Generates full-day trip itineraries based on   |
|   mood, interests, and budget                    |
|--------------------------------------------------|
|  🔧 Tools:                                       |
|   - Google Search                                |
|--------------------------------------------------|
|  🧠 Capabilities:                                |
|   - Budget Awareness (cheap / splurge)           |
|   - Mood Matching (adventurous, relaxing, etc.)  |
|   - Real-Time Info (hours, events)               |
|   - Morning / Afternoon / Evening plan           |
+--------------------------------------------------+

            ▲
            |
    +------------------+
    |   User Input     |
    |------------------|
    |  Mood            |
    |  Interests       |
    |  Budget          |
    +------------------+

            |
            ▼

+--------------------------------------------------+
|             Output: Markdown Itinerary           |
|--------------------------------------------------|
| - Time blocks (Morning / Afternoon / Evening)    |
| - Venue names with links and hours               |
| - Budget-matching activities                     |
+--------------------------------------------------+
```


In [4]:
# --- Agent Definition ---

def create_day_trip_agent():
    """Create the Spontaneous Day Trip Generator agent"""
    return Agent(
        name="day_trip_agent",
        model="gemini-2.5-flash",
        description="Agent specialized in generating spontaneous full-day itineraries based on mood, interests, and budget.",
        instruction="""
        You are the "Spontaneous Day Trip" Generator 🚗 - a specialized AI assistant that creates engaging full-day itineraries.

        Your Mission:
        Transform a simple mood or interest into a complete day-trip adventure with real-time details, while respecting a budget.

        Guidelines:
        1. **Budget-Aware**: Pay close attention to budget hints like 'cheap', 'affordable', or 'splurge'. Use Google Search to find activities (free museums, parks, paid attractions) that match the user's budget.
        2. **Full-Day Structure**: Create morning, afternoon, and evening activities.
        3. **Real-Time Focus**: Search for current operating hours and special events.
        4. **Mood Matching**: Align suggestions with the requested mood (adventurous, relaxing, artsy, etc.).

        RETURN itinerary in MARKDOWN FORMAT with clear time blocks and specific venue names.
        """,
        tools=[google_search]
    )

day_trip_agent = create_day_trip_agent()
print(f"🧞 Agent '{day_trip_agent.name}' is created and ready for adventure!")

🧞 Agent 'day_trip_agent' is created and ready for adventure!


In [6]:
# --- A Helper Function to Run Our Agents ---
# We'll use this function throughout the notebook to make running queries easy.

async def run_agent_query(agent: Agent, query: str, session: Session, user_id: str, is_router: bool = False):
    """Initializes a runner and executes a query for a given agent and session."""
    print(f"\n🚀 Running query for agent: '{agent.name}' in session: '{session.id}'...")

    runner = Runner(
        agent=agent,
        session_service=session_service,
        app_name=agent.name
    )

    final_response = ""
    try:
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session.id,
            new_message=Content(parts=[Part(text=query)], role="user")
        ):
            if not is_router:
                # Let's see what the agent is thinking!
                print(f"EVENT: {event}")
            if event.is_final_response():
                final_response = event.content.parts[0].text
    except Exception as e:
        final_response = f"An error occurred: {e}"

    if not is_router:
     print("\n" + "-"*50)
     print("✅ Final Response:")
     display(Markdown(final_response))
     print("-"*50 + "\n")

    return final_response

# --- Initialize our Session Service ---
# This one service will manage all the different sessions in our notebook.
session_service = InMemorySessionService()
my_user_id = "adk_adventurer_001"

In [7]:
# --- Let's test the Day Trip Genie! ---

async def run_day_trip_genie():
    # Create a new, single-use session for this query
    day_trip_session = await session_service.create_session(
        app_name=day_trip_agent.name,
        user_id=my_user_id
    )

    # Note the new budget constraint in the query!
    query = "Plan a relaxing and artsy day trip near Sunnyvale, CA. Keep it affordable!"
    print(f"🗣️ User Query: '{query}'")

    await run_agent_query(day_trip_agent, query, day_trip_session, my_user_id)

await run_day_trip_genie()

🗣️ User Query: 'Plan a relaxing and artsy day trip near Sunnyvale, CA. Keep it affordable!'

🚀 Running query for agent: 'day_trip_agent' in session: '3c30a4f2-60fc-4996-a6a3-4d303fc94e44'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Here's a relaxing and artsy day trip itinerary near Sunnyvale, CA, designed to be affordable for Sunday, July 5, 2026:

## Relaxing & Artsy Day Trip near Sunnyvale, CA (Affordable)

This itinerary focuses on free world-class art, serene outdoor spaces, and affordable local dining to give you a perfect blend of relaxation and cultural enrichment without breaking the bank.

---

### **Morning (10:00 AM - 1:00 PM): Immerse in Free World-Class Art**

*   **10:00 AM - 1:00 PM: Explore Cantor Arts Center & Anderson Collection at Stanford University (Palo Alto)**
    Start your day with a visit to Stanford University, home to two magnificent art museums offering free admission.
    *   **Cantor Arts Center:** Open 1

Here's a relaxing and artsy day trip itinerary near Sunnyvale, CA, designed to be affordable for Sunday, July 5, 2026:

## Relaxing & Artsy Day Trip near Sunnyvale, CA (Affordable)

This itinerary focuses on free world-class art, serene outdoor spaces, and affordable local dining to give you a perfect blend of relaxation and cultural enrichment without breaking the bank.

---

### **Morning (10:00 AM - 1:00 PM): Immerse in Free World-Class Art**

*   **10:00 AM - 1:00 PM: Explore Cantor Arts Center & Anderson Collection at Stanford University (Palo Alto)**
    Start your day with a visit to Stanford University, home to two magnificent art museums offering free admission.
    *   **Cantor Arts Center:** Open 10:00 AM - 5:00 PM on Sundays, the Cantor Arts Center boasts a diverse collection spanning 5,000 years of art, including a renowned collection of Rodin bronze sculptures displayed both inside and in its outdoor garden. It's a fantastic place for quiet contemplation and appreciating art from around the globe.
    *   **Anderson Collection at Stanford University:** Adjacent to the Cantor, this museum is dedicated to modern and contemporary American art, featuring works from the New York School, Bay Area Figuration, and California Light & Space movements. It's open 11:00 AM - 5:00 PM on Sundays and always free to visit.
    *   **Cost:** Free admission to both museums. Parking is also free on weekends at Stanford University.
    *   **Relaxation Factor:** Stroll through galleries at your own pace, enjoy the peaceful atmosphere, and take in the beauty of the Rodin Sculpture Garden.

### **Lunch (1:00 PM - 2:30 PM): Affordable Bites in Sunnyvale**

*   **1:00 PM - 2:30 PM: Grab an Affordable Lunch in Sunnyvale**
    Head back towards Sunnyvale for a budget-friendly and delicious lunch.
    *   **Option 1: Eighty Eight Sushi (Japanese Cuisine):** Located minutes from downtown Sunnyvale, they offer satisfying lunch options at competitive prices, including sushi rolls, ramen bowls, and bento boxes.
    *   **Option 2: Local Food Trucks (Mexican/Indian):** Sunnyvale is known for its excellent and affordable food trucks. Consider options like Cato's Tacos, El Califas Taco Truck, or Taqueria Patzcuaro for authentic and inexpensive Mexican food. For Indian, Apni Mandi offers an Indian plate with curries, rotis, and rice for around $10.
    *   **Cost:** ~$10-$20 per person.

### **Afternoon (2:30 PM - 5:30 PM): Public Art & Park Serenity**

*   **2:30 PM - 4:00 PM: Discover Public Art at Fair Oaks Park (Sunnyvale)**
    Sunnyvale has a vibrant public art scene. Fair Oaks Park is a great spot to enjoy both art and nature.
    *   The park features an "innovation zone" with interactive art and is home to some of the "Sun Flair" sculptures – 26 identical sun sculptures transformed by local artists, displayed in city parks through early 2027. You can also find the "Sunnyvale Community Mural" here.
    *   **Cost:** Free.
    *   **Relaxation Factor:** Enjoy a leisurely walk through the park, discover unique sculptures, and perhaps find a peaceful spot to sit and relax.

*   **4:00 PM - 5:30 PM: Unwind at Sunnyvale Baylands Park (Sunnyvale)**
    For more relaxation and natural beauty, head to Sunnyvale Baylands Park, a serene natural oasis.
    *   The park offers expansive open spaces, picturesque walking trails, and scenic picnic areas, perfect for birdwatching or simply enjoying the outdoors. You can access a section of the San Francisco Bay Trail from here.
    *   **Cost:** Free.
    *   **Relaxation Factor:** Take a gentle stroll, enjoy the bay breezes, and observe local wildlife.

### **Evening (6:00 PM onwards): Casual Dinner & Downtown Stroll**

*   **6:00 PM onwards: Relaxed Dinner in Downtown Sunnyvale**
    Conclude your day with a relaxing dinner in downtown Sunnyvale, which offers a variety of affordable dining options.
    *   Murphy Street in Downtown Sunnyvale has many restaurants with a pleasant sit-down vibe. Explore the options, from diverse international cuisines to casual American fare, to find something that suits your taste.
    *   **Cost:** ~$15-$30 per person, depending on your choice.
    *   **Relaxation Factor:** Enjoy a pleasant meal, then take a leisurely evening stroll through the quaint and quiet downtown area.

Enjoy your relaxing and artsy day trip!

--------------------------------------------------



---
## Part 2: Supercharging Agents with Custom Tools 🛠️

So far, we've used the powerful built-in `GoogleSearch` tool. But the true power of agents comes from connecting them to your own logic and data sources.

This is where **custom tools** come in. Let's explore three patterns for giving your agent new skills, using real-world, practical examples.

### 2.1 The Simple `FunctionTool`: Calling a Real-Time Weather API

The most direct way to create a tool is by writing a Python function. This is perfect for synchronous tasks like fetching data from an API.

**Key Concept:** The function's **docstring** is critical. The ADK uses it as the tool's official description, which the LLM reads to understand its purpose, parameters, and when to use it.

In this example, we'll create a tool that calls the **free, public U.S. National Weather Service API** to get a real-time forecast. No API key needed!

In [8]:
# --- Tool Definition: A function that calls a live public API ---
import requests
import json

# A simple lookup to avoid needing a separate geocoding API for this example
LOCATION_COORDINATES = {
    "sunnyvale": "37.3688,-122.0363",
    "san francisco": "37.7749,-122.4194",
    "lake tahoe": "39.0968,-120.0324"
}

def get_live_weather_forecast(location: str) -> dict:
    """Gets the current, real-time weather forecast for a specified location in the US.

    Args:
        location: The city name, e.g., "San Francisco".

    Returns:
        A dictionary containing the temperature and a detailed forecast.
    """
    print(f"🛠️ TOOL CALLED: get_live_weather_forecast(location='{location}')")

    # Find coordinates for the location
    normalized_location = location.lower()
    coords_str = None
    for key, val in LOCATION_COORDINATES.items():
        if key in normalized_location:
            coords_str = val
            break
    if not coords_str:
        return {"status": "error", "message": f"I don't have coordinates for {location}."}

    try:
        # NWS API requires 2 steps: 1. Get the forecast URL from the coordinates.
        points_url = f"https://api.weather.gov/points/{coords_str}"
        headers = {"User-Agent": "ADK Example Notebook"}
        points_response = requests.get(points_url, headers=headers)
        points_response.raise_for_status() # Raise an exception for bad status codes
        forecast_url = points_response.json()['properties']['forecast']

        # 2. Get the actual forecast from the URL.
        forecast_response = requests.get(forecast_url, headers=headers)
        forecast_response.raise_for_status()

        # Extract the relevant forecast details
        current_period = forecast_response.json()['properties']['periods'][0]
        return {
            "status": "success",
            "temperature": f"{current_period['temperature']}°{current_period['temperatureUnit']}",
            "forecast": current_period['detailedForecast']
        }
    except requests.exceptions.RequestException as e:
        return {"status": "error", "message": f"API request failed: {e}"}

# --- Agent Definition: An agent that USES the new tool ---

weather_agent = Agent(
    name="weather_aware_planner",
    model="gemini-2.5-flash",
    description="A trip planner that checks the real-time weather before making suggestions.",
    instruction="You are a cautious trip planner. Before suggesting any outdoor activities, you MUST use the `get_live_weather_forecast` tool to check conditions. Incorporate the live weather details into your recommendation.",
    tools=[get_live_weather_forecast]
)

print(f"🌦️ Agent '{weather_agent.name}' is created and can now call a live weather API!")

🌦️ Agent 'weather_aware_planner' is created and can now call a live weather API!


In [9]:
# --- Let's test the Weather-Aware Planner ---

async def run_weather_planner_test():
    weather_session = await session_service.create_session(app_name=weather_agent.name, user_id=my_user_id)
    query = "I want to go hiking near Lake Tahoe, what's the weather like?"
    print(f"🗣️ User Query: '{query}'")
    await run_agent_query(weather_agent, query, weather_session, my_user_id)

await run_weather_planner_test()

🗣️ User Query: 'I want to go hiking near Lake Tahoe, what's the weather like?'

🚀 Running query for agent: 'weather_aware_planner' in session: 'be0e293d-20b7-4f7e-988d-8feef5d3a711'...


/usr/local/lib/python3.12/dist-packages/google/adk/tools/function_tool.py:95: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  build_function_declaration(


EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'location': 'Lake Tahoe'
        },
        id='adk-a2df18ae-bff0-4881-a3db-93125f4775c6',
        name='get_live_weather_forecast'
      ),
      thought_signature=b'\n\xbe\x02\x01\x11M2\x0f}\xf0l~\xa22\xeav\x0e7\xb8\xef+b\x9f\xe4\xb5\xaf\xb1\xea\x9d\xed\xba\xddm?,\xe61\xb9\xd6KZ\xd5\xda\xe0{B`)@\xad"\x86\x19\x953\x18D\xf3\xccG)\x1e\x107\xfb\xea\xcb\xb4\x9d\x05\x8e)6P\xdcZ\xc8\xbf\x1e\x01\xbe\xc2<uc\xc2\xfb\xd6V\x95\x16g\xc9\xb1\xb1.L...'
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None turn_complete_reason=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=20,
  prompt_token_count=185,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 

The weather near Lake Tahoe is mostly cloudy with a temperature of 53°F. There's a west wind around 5 mph. It seems like a decent day for a hike, but you might want to bring layers and be prepared for cloudy conditions.

--------------------------------------------------



## 2.2 The Agent-as-a-Tool: Consulting a Specialist 🧑‍🍳

Why build one agent that does everything when you can build a **team of specialist agents?** The **Agent-as-a-Tool** pattern allows one agent to delegate a task to another agent.

**Key Concept:** This is different from a sub-agent. When Agent A calls Agent B as a tool, Agent B's response is passed **back to Agent A**. Agent A then uses that information to form its own final response to the user. It's a powerful way to compose complex behaviors from simpler, focused, and reusable agents.

### How It Works

Our top-level agent, the `trip_data_concierge_agent`, acts as the **Orchestrator**. It has two tools at its disposal:

1.  `call_db_agent`: A function that internally calls our `db_agent` to fetch raw data.
2.  `call_concierge_agent`: A function that calls the `concierge_agent`.

The `concierge_agent` itself has a tool: the `food_critic_agent`.

The flow for a complex query is:

1.  **User** asks the `trip_data_concierge_agent` for a hotel and a nearby restaurant.
2.  The **Orchestrator** first calls `call_db_agent` to get hotel data.
3.  The data is saved in `tool_context.state`.
4.  The **Orchestrator** then calls `call_concierge_agent`, which retrieves the hotel data from the context.
5.  The `concierge_agent` receives the request and decides it needs to use its own tool, the `food_critic_agent`.
6.  The `food_critic_agent` provides a witty recommendation.
7.  The `concierge_agent` gets the critic's response and politely formats it.
8.  This final, polished response is returned to the **Orchestrator**, which presents it to the user.

                         +-----------------------------------------------------------+
                         |              🧭 Trip Data Concierge Agent                 |
                         |-----------------------------------------------------------|
                         |  Model: gemini-2.5-flash                                  |
                         |  Description:                                             |
                         |   Orchestrates database query and travel recommendation  |
                         |-----------------------------------------------------------|
                         |  🔧 Tools:                                                |
                         |   1. call_db_agent                                        |
                         |   2. call_concierge_agent                                 |
                         +-----------------------------------------------------------+
                                      /                                \
                                     /                                  \
                                    ▼                                    ▼
        +-------------------------------------------+    +---------------------------------------------+
        |            🔧 Tool: call_db_agent         |    |         🔧 Tool: call_concierge_agent        |
        |-------------------------------------------|    |---------------------------------------------|
        | Calls: db_agent                           |    | Calls: concierge_agent                       |
        |                                           |    | Uses data from db_agent for recommendations |
        +-------------------------------------------+    +---------------------------------------------+
                                |                                          |
                                ▼                                          ▼
       +--------------------------------------------+   +------------------------------------------------+
       |              📦 db_agent                   |   |             🤵 concierge_agent                  |
       |--------------------------------------------|   |------------------------------------------------|
       | Model: gemini-2.5-flash                    |   | Model: gemini-2.5-flash                         |
       | Role: Return mock JSON hotel data          |   | Role: Hotel staff that handles user Q&A        |
       +--------------------------------------------+   | Tools:                                          |
                                                         |  - food_critic_agent                           |
                                                         +------------------------------------------------+
                                                                                 |
                                                                                 ▼
                                                       +------------------------------------------------+
                                                       |          🍽️ food_critic_agent                  |
                                                       |------------------------------------------------|
                                                       | Model: gemini-2.5-flash                         |
                                                       | Role: Gives a witty restaurant recommendation   |
                                                       +------------------------------------------------+


In [10]:
import asyncio
from google.adk.tools import ToolContext
from google.adk.tools.agent_tool import AgentTool

# Assume 'db_agent' is a pre-defined NL2SQL Agent
# For this example, we'll create placeholder agents.

db_agent = Agent(
    name="db_agent",
    model="gemini-2.5-flash",
    instruction="You are a database agent. When asked for data, return this mock JSON object: {'status': 'success', 'data': [{'name': 'The Grand Hotel', 'rating': 5, 'reviews': 450}, {'name': 'Seaside Inn', 'rating': 4, 'reviews': 620}]}")

# --- 1. Define the Specialist Agents ---

# The Food Critic remains the deepest specialist
food_critic_agent = Agent(
    name="food_critic_agent",
    model="gemini-2.5-flash",
    instruction="You are a snobby but brilliant food critic. You ONLY respond with a single, witty restaurant suggestion near the provided location.",
)

# The Concierge knows how to use the Food Critic
concierge_agent = Agent(
    name="concierge_agent",
    model="gemini-2.5-flash",
    instruction="You are a five-star hotel concierge. If the user asks for a restaurant recommendation, you MUST use the `food_critic_agent` tool. Present the opinion to the user politely.",
    tools=[AgentTool(agent=food_critic_agent)]
)


# --- 2. Define the Tools for the Orchestrator ---

async def call_db_agent(
    question: str,
    tool_context: ToolContext,
):
    """
    Use this tool FIRST to connect to the database and retrieve a list of places, like hotels or landmarks.
    """
    print("--- TOOL CALL: call_db_agent ---")
    agent_tool = AgentTool(agent=db_agent)
    db_agent_output = await agent_tool.run_async(
        args={"request": question}, tool_context=tool_context
    )
    # Store the retrieved data in the context's state
    tool_context.state["retrieved_data"] = db_agent_output
    return db_agent_output


async def call_concierge_agent(
    question: str,
    tool_context: ToolContext,
):
    """
    After getting data with call_db_agent, use this tool to get travel advice, opinions, or recommendations.
    """
    print("--- TOOL CALL: call_concierge_agent ---")
    # Retrieve the data fetched by the previous tool
    input_data = tool_context.state.get("retrieved_data", "No data found.")

    # Formulate a new prompt for the concierge, giving it the data context
    question_with_data = f"""
    Context: The database returned the following data: {input_data}

    User's Request: {question}
    """

    agent_tool = AgentTool(agent=concierge_agent)
    concierge_output = await agent_tool.run_async(
        args={"request": question_with_data}, tool_context=tool_context
    )
    return concierge_output


# --- 3. Define the Top-Level Orchestrator Agent ---

trip_data_concierge_agent = Agent(
    name="trip_data_concierge",
    model="gemini-2.5-flash",
    description="Top-level agent that queries a database for travel data, then calls a concierge agent for recommendations.",
    tools=[call_db_agent, call_concierge_agent],
    instruction="""
    You are a master travel planner who uses data to make recommendations.

    1.  **ALWAYS start with the `call_db_agent` tool** to fetch a list of places (like hotels) that match the user's criteria.

    2.  After you have the data, **use the `call_concierge_agent` tool** to answer any follow-up questions for recommendations, opinions, or advice related to the data you just found.
    """,
)

print(f"✅ Orchestrator Agent '{trip_data_concierge_agent.name}' is defined and ready.")

✅ Orchestrator Agent 'trip_data_concierge' is defined and ready.


In [11]:
# --- Let's test the Trip Data Concierge Agent ---

async def run_trip_data_concierge():
    """
    Sets up a session and runs a query against the top-level
    trip_data_concierge_agent.
    """
    # Create a new, single-use session for this query
    concierge_session = await session_service.create_session(
        app_name=trip_data_concierge_agent.name,
        user_id=my_user_id
    )

    # This query is specifically designed to trigger the full two-step process:
    # 1. Get data from the db_agent.
    # 2. Get a recommendation from the concierge_agent based on that data.
    query = "Find the top-rated hotels in San Francisco from the database, then suggest a dinner spot near the one with the most reviews."
    print(f"🗣️ User Query: '{query}'")

    # We call our existing helper function with the top-level orchestrator agent
    await run_agent_query(trip_data_concierge_agent, query, concierge_session, my_user_id)

# Run the test
await run_trip_data_concierge()

🗣️ User Query: 'Find the top-rated hotels in San Francisco from the database, then suggest a dinner spot near the one with the most reviews.'

🚀 Running query for agent: 'trip_data_concierge' in session: '56a6b374-d5b3-4616-9938-e703bfedbe96'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'question': 'Find the top-rated hotels in San Francisco'
        },
        id='adk-9414003a-5683-4816-82fc-4db01dd1012c',
        name='call_db_agent'
      ),
      thought_signature=b'\n\x8b\x04\x01\x11M2\x0f\x7f\xffX\xe1\xd0\xc9\xabD\x05\x8bb\x0f\x99\x96k\x81\x81\xb3/\xe9\xba\x05\xe1\x03\xfa\x99mE\xf9\xd1a\x1e\x1f\xd4\x1fb\xf4~~)\x07\x884LQ:\x91n\xb6Q\xda\xf2\x07U\xe0\xda\x1d\xe1\x01=\xb3\xa6\x14\xcb\xb6\xbd\x8a\xde\xf0C\xa4\xc7\xdb\xc7\xf4\xd1\xfd1\x8c\xca\x9a\x9e}\xec\x90\xe8\x1e\xb6\xe4...'
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None turn_complete_reason=None fini

While I couldn't filter hotels by San Francisco specifically, I found two hotels in the database: "The Grand Hotel" (5-star, 450 reviews) and "Seaside Inn" (4-star, 620 reviews).

"Seaside Inn" has the most reviews. Although I don't have location data to suggest a dinner spot *near* it, the food critic suggests "The Gilded Sardine. Naturally."

--------------------------------------------------



---
## Part 3: Agent with a Memory - The Adaptive Planner 🗺️

Now, let's see an agent that not only **remembers** but also **adapts**. We'll challenge the `multi_day_trip_agent` to re-plan part of its itinerary based on our feedback. This is a much more realistic test of conversational AI.

```
+-----------------------------------------------------+
|         Adaptive Multi-Day Trip Agent 🗺️           |
|-----------------------------------------------------|
|  Model: gemini-2.5-flash                            |
|  Description:                                       |
|   Builds multi-day travel itineraries step-by-step, |
|   remembers previous days, adapts to feedback       |
|-----------------------------------------------------|
|  🔧 Tools:                                          |
|   - Google Search                                   |
|-----------------------------------------------------|
|  🧠 Capabilities:                                   |
|   - Memory of past conversation & preferences       |
|   - Progressive planning (1 day at a time)          |
|   - Adapts to user feedback                         |
|   - Ensures activity variety across days            |
+-----------------------------------------------------+

            ▲
            |
    +---------------------------+
    |     User Interaction      |
    |---------------------------|
    | - Destination             |
    | - Trip duration           |
    | - Interests & feedback    |
    +---------------------------+

            |
            ▼

+-----------------------------------------------------+
|        Day-by-Day Itinerary Generation              |
|-----------------------------------------------------|
|  🗓️ Day N Output (Markdown format):                 |
|   - Morning / Afternoon / Evening activities        |
|   - Personalized & context-aware                    |
|   - Changes accepted, feedback acknowledged         |
+-----------------------------------------------------+

            |
            ▼

+-----------------------------------------------------+
|        Next Day Planning Triggered 🚀               |
|-----------------------------------------------------|
| - Builds on prior days                              |
| - Avoids repetition                                 |
| - Asks user for confirmation before proceeding      |
+-----------------------------------------------------+
```


In [12]:
# --- Agent Definition: The Adaptive Planner ---

def create_multi_day_trip_agent():
    """Create the Progressive Multi-Day Trip Planner agent"""
    return Agent(
        name="multi_day_trip_agent",
        model="gemini-2.5-flash",
        description="Agent that progressively plans a multi-day trip, remembering previous days and adapting to user feedback.",
        instruction="""
        You are the "Adaptive Trip Planner" 🗺️ - an AI assistant that builds multi-day travel itineraries step-by-step.

        Your Defining Feature:
        You have short-term memory. You MUST refer back to our conversation to understand the trip's context, what has already been planned, and the user's preferences. If the user asks for a change, you must adapt the plan while keeping the unchanged parts consistent.

        Your Mission:
        1.  **Initiate**: Start by asking for the destination, trip duration, and interests.
        2.  **Plan Progressively**: Plan ONLY ONE DAY at a time. After presenting a plan, ask for confirmation.
        3.  **Handle Feedback**: If a user dislikes a suggestion (e.g., "I don't like museums"), acknowledge their feedback, and provide a *new, alternative* suggestion for that time slot that still fits the overall theme.
        4.  **Maintain Context**: For each new day, ensure the activities are unique and build logically on the previous days. Do not suggest the same things repeatedly.
        5.  **Final Output**: Return each day's itinerary in MARKDOWN format.
        """,
        tools=[google_search]
    )

multi_day_agent = create_multi_day_trip_agent()
print(f"🗺️ Agent '{multi_day_agent.name}' is created and ready to plan and adapt!")

🗺️ Agent 'multi_day_trip_agent' is created and ready to plan and adapt!


### Scenario 3a: Agent WITH Memory (Using a SINGLE Session) ✅

First, let's see the correct way to do it. We will use the **exact same `trip_session` object** for the entire conversation. Watch how the agent remembers the context from Turn 1 to correctly handle the requests in Turn 2 and 3.

In [13]:
# --- Scenario 2: Testing Adaptation and Memory ---

async def run_adaptive_memory_demonstration():
    print("### 🧠 DEMO 2: AGENT THAT ADAPTS (SAME SESSION) ###")

    # Create ONE session that we will reuse for the whole conversation
    trip_session = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"Created a single session for our trip: {trip_session.id}")

    # --- Turn 1: The user initiates the trip ---
    query1 = "Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food."
    print(f"\n🗣️ User (Turn 1): '{query1}'")
    await run_agent_query(multi_day_agent, query1, trip_session, my_user_id)

    # --- Turn 2: The user gives FEEDBACK and asks for a CHANGE ---
    # We use the EXACT SAME `trip_session` object!
    query2 = "That sounds pretty good, but I'm not a huge fan of castles. Can you replace the morning activity for Day 1 with something else historical?"
    print(f"\n🗣️ User (Turn 2 - Feedback): '{query2}'")
    await run_agent_query(multi_day_agent, query2, trip_session, my_user_id)

    # --- Turn 3: The user confirms and asks to continue ---
    query3 = "Yes, the new plan for Day 1 is perfect! Please plan Day 2 now, keeping the food theme in mind."
    print(f"\n🗣️ User (Turn 3 - Confirmation): '{query3}'")
    await run_agent_query(multi_day_agent, query3, trip_session, my_user_id)

await run_adaptive_memory_demonstration()

### 🧠 DEMO 2: AGENT THAT ADAPTS (SAME SESSION) ###
Created a single session for our trip: d6fb7f27-4ae5-4145-a71e-177de01e562b

🗣️ User (Turn 1): 'Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: 'd6fb7f27-4ae5-4145-a71e-177de01e562b'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Okay, fantastic! Lisbon is a wonderful choice for history and food lovers. Let's start planning your first day.

Here's a possible itinerary for **Day 1**:

### Day 1: Alfama's Charms & Castle Views

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama District & São Jorge Castle**
    *   Begin your day by wandering through the labyrinthine streets of **Alfama**, Lisbon's oldest district. Discover hidden squares, charming alleys, and historic churches.
    *   Ascend to the majestic **São Jorge Castle (Castelo de São Jorge)**, a Moorish castle 

Okay, fantastic! Lisbon is a wonderful choice for history and food lovers. Let's start planning your first day.

Here's a possible itinerary for **Day 1**:

### Day 1: Alfama's Charms & Castle Views

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama District & São Jorge Castle**
    *   Begin your day by wandering through the labyrinthine streets of **Alfama**, Lisbon's oldest district. Discover hidden squares, charming alleys, and historic churches.
    *   Ascend to the majestic **São Jorge Castle (Castelo de São Jorge)**, a Moorish castle offering panoramic views over the city and the Tagus River. Explore its ramparts, gardens, and archaeological site.
*   **Lunch (1:00 PM - 2:30 PM): Traditional Portuguese Lunch in Alfama**
    *   Enjoy a traditional Portuguese meal at a local tasca in Alfama, perhaps trying "Bacalhau à Brás" (shredded cod with onions, potatoes, and scrambled eggs) or grilled sardines.
*   **Afternoon (2:30 PM - 6:00 PM): Lisbon Cathedral & Baixa District**
    *   Visit the **Lisbon Cathedral (Sé de Lisboa)**, the city's oldest church, showcasing a blend of Romanesque, Gothic, and Baroque styles.
    *   Stroll through the **Baixa district**, rebuilt after the 1755 earthquake with its elegant grid of streets, grand squares like Praça do Comércio, and neoclassical architecture.
*   **Evening (7:00 PM onwards): Dinner in Chiado & Fado Experience**
    *   Head to the sophisticated **Chiado district** for dinner at one of its many acclaimed restaurants.
    *   Consider experiencing a traditional **Fado show** in a local restaurant or "Casa de Fado" in Alfama or Bairro Alto, an integral part of Portuguese culture.

How does this sound for your first day in Lisbon? We can adjust anything you like!

--------------------------------------------------


🗣️ User (Turn 2 - Feedback): 'That sounds pretty good, but I'm not a huge fan of castles. Can you replace the morning activity for Day 1 with something else historical?'

🚀 Running query for agent: 'multi_day_trip_agent' in session: 'd6fb7f27-4ae5-4145-a71e-177de01e562b'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Understood! No problem at all, we can definitely swap out the castle for another historical gem.

Here's a revised itinerary for **Day 1** with a new historical focus for the morning:

### Day 1: Alfama's Charms & Ancient History

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama District & Lisbon's Roman Theatre Museum**
    *   Begin your day by wandering through the labyrinthine streets of **Alfama**, Lisbon's oldest district. Discover hidden squares, charming alleys, and historic churches.
    *   Dive deeper into Lisbon's ancient past with a visit to the **Roman Theatre

Understood! No problem at all, we can definitely swap out the castle for another historical gem.

Here's a revised itinerary for **Day 1** with a new historical focus for the morning:

### Day 1: Alfama's Charms & Ancient History

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama District & Lisbon's Roman Theatre Museum**
    *   Begin your day by wandering through the labyrinthine streets of **Alfama**, Lisbon's oldest district. Discover hidden squares, charming alleys, and historic churches.
    *   Dive deeper into Lisbon's ancient past with a visit to the **Roman Theatre Museum (Teatro Romano)**. Located within Alfama, you can explore the ruins of an ancient Roman theatre and learn about its history through exhibits.
*   **Lunch (1:00 PM - 2:30 PM): Traditional Portuguese Lunch in Alfama**
    *   Enjoy a traditional Portuguese meal at a local tasca in Alfama, perhaps trying "Bacalhau à Brás" (shredded cod with onions, potatoes, and scrambled eggs) or grilled sardines.
*   **Afternoon (2:30 PM - 6:00 PM): Lisbon Cathedral & Baixa District**
    *   Visit the **Lisbon Cathedral (Sé de Lisboa)**, the city's oldest church, showcasing a blend of Romanesque, Gothic, and Baroque styles.
    *   Stroll through the **Baixa district**, rebuilt after the 1755 earthquake with its elegant grid of streets, grand squares like Praça do Comércio, and neoclassical architecture.
*   **Evening (7:00 PM onwards): Dinner in Chiado & Fado Experience**
    *   Head to the sophisticated **Chiado district** for dinner at one of its many acclaimed restaurants.
    *   Consider experiencing a traditional **Fado show** in a local restaurant or "Casa de Fado" in Alfama or Bairro Alto, an integral part of Portuguese culture.

How does this revised plan for Day 1 sound?

--------------------------------------------------


🗣️ User (Turn 3 - Confirmation): 'Yes, the new plan for Day 1 is perfect! Please plan Day 2 now, keeping the food theme in mind.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: 'd6fb7f27-4ae5-4145-a71e-177de01e562b'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Excellent! I'm glad Day 1 is perfect. Let's move on to an exciting **Day 2**, keeping your interest in historic sites and great local food front and center.

Here's a possible itinerary for **Day 2**:

### Day 2: Maritime History & Culinary Delights

*   **Morning (9:00 AM - 1:00 PM): Belém's Age of Discovery**
    *   Start your day in the historic **Belém district**, home to monuments commemorating Portugal's Age of Discoveries.
    *   Visit the magnificent **Jerónimos Monastery (Mosteiro dos Jerónimos)**, a UNESCO World Heritage site and a masterpiece of Manueline architecture, where Vasco da Gama is entombed.

Excellent! I'm glad Day 1 is perfect. Let's move on to an exciting **Day 2**, keeping your interest in historic sites and great local food front and center.

Here's a possible itinerary for **Day 2**:

### Day 2: Maritime History & Culinary Delights

*   **Morning (9:00 AM - 1:00 PM): Belém's Age of Discovery**
    *   Start your day in the historic **Belém district**, home to monuments commemorating Portugal's Age of Discoveries.
    *   Visit the magnificent **Jerónimos Monastery (Mosteiro dos Jerónimos)**, a UNESCO World Heritage site and a masterpiece of Manueline architecture, where Vasco da Gama is entombed.
    *   Stroll along the Tagus River to admire the iconic **Belém Tower (Torre de Belém)**, a fortress that once guarded the city, and the grand **Monument to the Discoveries (Padrão dos Descobrimentos)**.
*   **Lunch (1:00 PM - 2:30 PM): The Original Pastéis de Belém & Local Flavors**
    *   No trip to Belém is complete without indulging in the warm, original custard tarts at **Fábrica de Pastéis de Belém**.
    *   Follow this sweet treat with a savory lunch at a traditional restaurant in the Belém area, perhaps sampling fresh seafood or other regional specialties.
*   **Afternoon (2:30 PM - 6:00 PM): Tram 28 Ride & Graça Views**
    *   Embark on a classic Lisbon experience with a ride on the famous **Tram 28**. This iconic tram winds through several historic neighborhoods, offering a unique perspective of the city's charming streets and hills.
    *   Consider hopping off in the **Graça district** to visit the Miradouro da Senhora do Monte for breathtaking panoramic views of Lisbon, São Jorge Castle, and the Tagus River.
*   **Evening (7:00 PM onwards): Dinner at Time Out Market (Mercado da Ribeira)**
    *   For your farewell dinner, immerse yourself in the vibrant atmosphere of the **Time Out Market (Mercado da Ribeira)**. This bustling food hall features a curated selection of stalls from some of Lisbon's best chefs and restaurants, offering a fantastic opportunity to try a wide variety of high-quality local dishes and international cuisine under one roof.

How does this plan for your second day in Lisbon sound?

--------------------------------------------------



### Scenario 3b: Agent WITHOUT Memory (Using SEPARATE Sessions) ❌

Now, let's see what happens if we mess up our session management. Here, we'll give the agent a case of amnesia by creating a **brand new, separate session for each turn**.

Pay close attention to the agent's response to the second query. Because it's in a new session, it has no memory of the trip to Lisbon we just discussed!

In [14]:
# --- Scenario 2b: Demonstrating Memory FAILURE ---

async def run_memory_failure_demonstration():
    print("\n" + "#"*60)
    print("### 🧠 DEMO 2b: AGENT WITH AMNESIA (SEPARATE SESSIONS) ###")
    print("#"*60)

    # --- Turn 1: The user initiates the trip in the FIRST session ---
    query1 = "Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food."
    session_one = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"\nCreated a session for Turn 1: {session_one.id}")
    print(f"🗣️ User (Turn 1): '{query1}'")
    await run_agent_query(multi_day_agent, query1, session_one, my_user_id)

    # --- Turn 2: The user asks to continue... but in a completely NEW session ---
    query2 = "Yes, that looks perfect! Please plan Day 2."
    session_two = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"\nCreated a BRAND NEW session for Turn 2: {session_two.id}")
    print(f"🗣️ User (Turn 2): '{query2}'")
    await run_agent_query(multi_day_agent, query2, session_two, my_user_id)

await run_memory_failure_demonstration()


############################################################
### 🧠 DEMO 2b: AGENT WITH AMNESIA (SEPARATE SESSIONS) ###
############################################################

Created a session for Turn 1: 1e4b999e-5813-4ce1-9f56-b612c783db75
🗣️ User (Turn 1): 'Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '1e4b999e-5813-4ce1-9f56-b612c783db75'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Okay, I can help you plan a fantastic 2-day trip to Lisbon focusing on historic sites and delicious local food!

Let's start with **Day 1**. How about this itinerary?

**Day 1: Alfama's Charm & Baixa's Grandeur**

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama District & São Jorge Castle**
    *   Begin your day in the historic Alfama district, Lisbon's oldest neighborhood. Wander through its narrow, winding streets, soa

Okay, I can help you plan a fantastic 2-day trip to Lisbon focusing on historic sites and delicious local food!

Let's start with **Day 1**. How about this itinerary?

**Day 1: Alfama's Charm & Baixa's Grandeur**

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama District & São Jorge Castle**
    *   Begin your day in the historic Alfama district, Lisbon's oldest neighborhood. Wander through its narrow, winding streets, soak in the traditional atmosphere, and discover hidden viewpoints.
    *   Visit the impressive **Lisbon Cathedral (Sé de Lisboa)**, a beautiful Romanesque cathedral with a rich history.
    *   Ascend to **São Jorge Castle (Castelo de São Jorge)**, a majestic Moorish castle offering panoramic views of the city and the Tagus River. Explore its ancient walls and peacocks roaming the grounds.
*   **Lunch (1:00 PM - 2:30 PM): Traditional Portuguese Meal in Alfama**
    *   Enjoy a traditional Portuguese lunch at a local tasca (tavern) in Alfama, savoring dishes like grilled sardines or bacalhau (codfish).
*   **Afternoon (2:30 PM - 6:00 PM): Baixa & Chiado Districts**
    *   Head to the Baixa district, rebuilt after the 1755 earthquake, known for its neoclassical architecture and grand squares like Praça do Comércio.
    *   Stroll through Rua Augusta, a bustling pedestrian street, and take the Santa Justa Lift for unique city views (optional, as São Jorge Castle offers great views).
    *   Continue to the elegant Chiado district, home to historic cafes, theaters, and charming shops.
*   **Evening (7:00 PM onwards): Dinner & Fado in Bairro Alto**
    *   Dine in the vibrant Bairro Alto district, known for its diverse restaurant scene.
    *   Consider experiencing a traditional Fado show (a Portuguese musical genre) at one of the local Fado houses for a truly authentic evening.

How does this sound for your first day in Lisbon?

--------------------------------------------------


Created a BRAND NEW session for Turn 2: 75294618-6d9d-44fc-903c-876b3f862c10
🗣️ User (Turn 2): 'Yes, that looks perfect! Please plan Day 2.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '75294618-6d9d-44fc-903c-876b3f862c10'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Okay, fantastic! I'm glad Day 1 looks perfect. Let's plan Day 2 to continue your adventure in Paris!

Here’s a suggestion for Day 2, focusing on a different iconic neighborhood and some historical gems:

### Day 2: Montmartre Charm & Latin Quarter History

*   **Morning (9:00 AM - 1:00 PM): Explore Montmartre & Sacré-Cœur**
    *   Begin your day with a visit to the charming hilltop neighborhood of Montmartre. Explore the stunning Sacré-Cœur Basilica, offering breathtaking panoramic views of Paris.
    *   Wander through the artistic Place du Tertre, where painters set up their easels, and soak in the b

Okay, fantastic! I'm glad Day 1 looks perfect. Let's plan Day 2 to continue your adventure in Paris!

Here’s a suggestion for Day 2, focusing on a different iconic neighborhood and some historical gems:

### Day 2: Montmartre Charm & Latin Quarter History

*   **Morning (9:00 AM - 1:00 PM): Explore Montmartre & Sacré-Cœur**
    *   Begin your day with a visit to the charming hilltop neighborhood of Montmartre. Explore the stunning Sacré-Cœur Basilica, offering breathtaking panoramic views of Paris.
    *   Wander through the artistic Place du Tertre, where painters set up their easels, and soak in the bohemian atmosphere.
    *   Discover the quieter streets and hidden gems of Montmartre, including the Wall of Love and the Dalida statue.
*   **Lunch (1:00 PM - 2:30 PM): Montmartre Bistro Experience**
    *   Enjoy a traditional French lunch at a cozy bistro or crêperie in the Montmartre area.
*   **Afternoon (2:30 PM - 6:30 PM): Île de la Cité & Latin Quarter Exploration**
    *   Head to Île de la Cité, the historical heart of Paris. See the exterior of the iconic Notre Dame Cathedral (currently under reconstruction).
    *   Visit the exquisite Sainte-Chapelle, renowned for its stunning stained-glass windows.
    *   Cross into the vibrant Latin Quarter. Stroll past the historic Sorbonne University and browse the famous Shakespeare and Company bookstore.
    *   Relax and unwind in the beautiful Luxembourg Gardens, admiring the Medici Fountain and the palace.
*   **Evening (7:00 PM onwards): Dinner & Jazz in the Latin Quarter**
    *   Dine at one of the many diverse restaurants in the lively Latin Quarter, known for its student-friendly atmosphere and wide range of cuisines.
    *   Optionally, enjoy some live music at a local jazz club or a drink at a traditional French café.

How does this sound for your second day?

--------------------------------------------------



See? The agent was confused! It likely asked what destination or what trip we were talking about. Because the second query was in a fresh, isolated session, the agent had no memory of planning Day 1 in Lisbon.

This perfectly illustrates why **managing sessions is the key to building truly conversational agents!**

---
## 🎉 Congratulations! 🎉

Congratulations on completing your ADK adventure into Tools and Memory! You've taken a massive leap from building single-shot agents to creating dynamic, stateful AI systems.

Let's recap the powerful concepts you've mastered:

- **Fundamental Agent & Tools**: You started by building a "Day Trip Genie" and equipped it with its first tool, GoogleSearch.

- **Custom Function Tools**: You gave your agent a new sense by creating a custom tool to fetch live data from the U.S. National Weather Service API.

- **Agent-as-a-Tool**: You orchestrated a sophisticated hierarchy where agents delegate tasks to other, more specialized agents, creating a collaborative team.

- **The Power of Memory**: Most importantly, you saw firsthand how managing a single, persistent Session allows an agent to remember context, adapt to user feedback, and conduct a meaningful, multi-turn conversation.

```
   __            /\_/\         /\_/\        /\_/\         __             (\__/)
o-''|\_____/).  ( o.o )       ( -.- )      ( ^_^ )     o-''|\_____/).    ( ^_^ )
 \_/|_)     )    > ^ <         > * <        >💖<         \_/|_)     )     / >🌸< \
    \  __  /                                              \  __  /         /   \
    (_/ (_/                                               (_/ (_/        (___|___)
```
